# ⚡ Thai Parasol: Dense MVS & High-Resolution Meshing on Kaggle

This notebook runs a complete **CUDA-accelerated Dense Multi-View Stereo (MVS)** pipeline on Kaggle GPU to reconstruct a watertight, razor-sharp 3D mesh from **[`dataset_neural.zip`](#)** (transparent RGBA PNGs with calibrated multi-ring camera poses).

### ⚙️ Required Kaggle Settings (Right Sidebar):
1. **Accelerator:** `GPU T4 x2` or `GPU P100` (Free 30 hrs/week)
2. **Internet:** **`On`** *(Crucial: needed to `pip install pycolmap-cuda12 open3d`)*
3. **Persistence:** `No persistence` or `Files only`

## 🔍 Step 1: Verify NVIDIA GPU & Install CUDA PyCOLMAP & Open3D

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

# 2. Install PyCOLMAP (CUDA 12) & Open3D (pin pillow<12.0 to prevent gradio / _typing conflict)
print("Installing CUDA-enabled PyCOLMAP & Open3D...")
!pip install -q "pillow<12.0" pycolmap-cuda12 open3d

# 3. Flush stale in-memory PIL submodules cached by IPython before pip install
import sys
for m in list(sys.modules.keys()):
    if m == "PIL" or m.startswith("PIL."):
        del sys.modules[m]

# 4. Verify imports
import torch, pycolmap, open3d as o3d
from PIL import Image
print(f" -> PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" -> GPU Device: {torch.cuda.get_device_name(0)}")
print(f" -> PyCOLMAP Version: {pycolmap.__version__}")
print(f" -> Open3D Version:   {o3d.__version__}")
print(" -> Dependencies successfully loaded!")

## 📦 Step 2: Locate and Unpack `dataset_neural.zip`
The code below searches `/kaggle/input/` and `/kaggle/working/` for `dataset_neural.zip` (or uncompressed dataset folders) and unpacks it into `/kaggle/working/dataset`.

In [ ]:
import os, shutil, glob, gc
from pathlib import Path
import numpy as np
from PIL import Image
import pycolmap
import open3d as o3d

working_dataset = Path("/kaggle/working/dataset")
if working_dataset.exists():
    shutil.rmtree(working_dataset)
working_dataset.mkdir(parents=True, exist_ok=True)

# Search for dataset archive
zip_candidates = glob.glob("/kaggle/input/**/dataset_neural*.zip", recursive=True) + \
                 glob.glob("/kaggle/input/**/colab_dataset*.zip", recursive=True) + \
                 glob.glob("/kaggle/input/**/*.zip", recursive=True) + \
                 glob.glob("/kaggle/working/*.zip")

if zip_candidates:
    target_zip = zip_candidates[0]
    print(f"📦 Unpacking dataset zip: {target_zip}")
    !unzip -q -o "{target_zip}" -d /kaggle/working/dataset
else:
    input_folders = glob.glob("/kaggle/input/**/images", recursive=True)
    if input_folders:
        src_root = Path(input_folders[0]).parent
        print(f"📦 Copying dataset from directory: {src_root}")
        shutil.copytree(src_root, working_dataset, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            "❌ No dataset found! Please click '+ Add Data' on the right sidebar "
            "and upload 'dataset_neural.zip'."
        )

# Handle nested root folder
if not (working_dataset / "images").exists():
    for sub in working_dataset.glob("**/*"):
        if (sub / "images").exists():
            for item in sub.iterdir():
                shutil.move(str(item), str(working_dataset))
            break

# Locate sparse model and image directory
def resolve_paths(base_dir=working_dataset):
    sparse_dirs = [p.parent for p in base_dir.rglob("points3D.bin")]
    if not sparse_dirs:
        raise FileNotFoundError("Could not find 'points3D.bin' in dataset.")
    
    best_sparse = sparse_dirs[0]
    max_pts = 0
    for s in sparse_dirs:
        f = s / "points3D.bin"
        if f.exists() and f.stat().st_size > max_pts:
            max_pts = f.stat().st_size
            best_sparse = s

    cv1_dirs = list(base_dir.rglob("cv-1"))
    if not cv1_dirs:
        img_dirs = list(base_dir.rglob("images"))
        if img_dirs:
            images_root = img_dirs[0]
        else:
            raise FileNotFoundError("Could not find 'cv-1' or 'images' directory.")
    else:
        images_root = cv1_dirs[0].parent

    return images_root, best_sparse

images_dir, sparse_dir = resolve_paths()
print(f" -> Images Directory: {images_dir}")
print(f" -> Sparse Model:     {sparse_dir}")

## 🎭 Step 3: Extract Binary Silhouette Masks from Transparent PNGs
Transparent PNGs contain an alpha channel ($A > 10$ foreground, $A = 0$ transparent background). We generate binary masks so that `stereo_fusion` guarantees **zero background points** in the dense cloud.

In [ ]:
dense_dir = Path("/kaggle/working/dense_workspace")
dense_dir.mkdir(parents=True, exist_ok=True)
masks_dir = dense_dir / "masks"
masks_dir.mkdir(parents=True, exist_ok=True)

rec = pycolmap.Reconstruction(sparse_dir)
print(f"Generating silhouette masks for {len(rec.images)} registered images...")

total_masks = 0
for img_id, img_obj in rec.images.items():
    src_png = images_dir / img_obj.name
    if src_png.exists():
        with Image.open(src_png) as pil_img:
            if pil_img.mode == "RGBA":
                alpha = np.array(pil_img)[:, :, 3]
                mask_arr = np.where(alpha > 10, 255, 0).astype(np.uint8)
            else:
                arr = np.array(pil_img)
                mask_arr = np.where(np.any(arr > 10, axis=-1), 255, 0).astype(np.uint8)
            
            mask_img = Image.fromarray(mask_arr)
            dst_mask_a = masks_dir / f"{img_obj.name}"
            dst_mask_b = masks_dir / f"{img_obj.name}.png"
            dst_mask_a.parent.mkdir(parents=True, exist_ok=True)
            mask_img.save(dst_mask_a)
            mask_img.save(dst_mask_b)
            total_masks += 1

print(f"✅ Successfully created {total_masks} silhouette masks in {masks_dir}")

## ⚡ Step 4: Run Dense Multi-View Stereo (Undistort $\rightarrow$ PatchMatch $\rightarrow$ Stereo Fusion)
- **Undistort:** Resizes and rectifies cameras.
- **PatchMatch Stereo:** Runs on GPU (`gpu_index="0"`) to calculate pixel-wise depth maps.
- **Stereo Fusion:** Merges depth maps into a dense 3D point cloud filtered by the alpha masks.

In [ ]:
# 1. Undistort images
print("1. Undistorting images for dense stereo matching...")
undistort_options = pycolmap.UndistortCameraOptions()
undistort_options.max_image_size = 1280

pycolmap.undistort_images(
    output_path=dense_dir,
    input_path=sparse_dir,
    image_path=images_dir,
    undistort_options=undistort_options,
)

# 2. Run CUDA PatchMatch Stereo on GPU (~8-12 mins on T4 GPU)
print("\n2. Running CUDA PatchMatch Stereo on GPU...")
pm_options = pycolmap.PatchMatchOptions(
    max_image_size=1280,
    geom_consistency=False,  # Photometric: 3x faster, cuts disk cache by 50%
    cache_size=6.0,          # Kaggle has 30 GB RAM (6 GB cache is optimal & fast)
    gpu_index="0",
    filter=True,
    filter_min_ncc=0.15,
    window_radius=5,
)
pycolmap.patch_match_stereo(
    workspace_path=dense_dir,
    options=pm_options
)

# 3. Run Stereo Fusion with Alpha Mask Guidance
print("\n3. Running Stereo Fusion with Alpha Mask Guidance...")
fused_ply = dense_dir / "dense_fused.ply"
fusion_options = pycolmap.StereoFusionOptions(
    max_image_size=1280,
    cache_size=6.0,
    min_num_pixels=3,
    mask_path=str(masks_dir)  # Zero points triangulated on transparent background
)
pycolmap.stereo_fusion(
    output_path=fused_ply,
    workspace_path=dense_dir,
    options=fusion_options,
    input_type="photometric",
    output_type="ply"
)
print(f"\n🎉 Dense Point Cloud Created: {fused_ply} ({os.path.getsize(fused_ply)/(1024*1024):.1f} MB)")

# ⚡ Purge temporary PatchMatch stereo bin cache (~8GB) to protect Kaggle disk limit (20GB)
stereo_cache_dir = dense_dir / "stereo"
if stereo_cache_dir.exists():
    print(" -> Purging temporary PatchMatch cache to free disk space...")
    shutil.rmtree(stereo_cache_dir)
gc.collect()

## 📐 Step 5: Screened Poisson Watertight Meshing & Taubin Smoothing
Converts the dense point cloud into a watertight, smooth polygon mesh with Taubin non-shrinking filtering to preserve fine bamboo ribs and sharp rims.

In [ ]:
print("Building High-Resolution Watertight Mesh...")

# Load dense point cloud
pcd = o3d.io.read_point_cloud(str(fused_ply))
print(f" -> Raw dense points: {len(pcd.points):,} (Colors: {pcd.has_colors()}, Normals: {pcd.has_normals()})")

# 1. Strip black border points from alpha edges
if pcd.has_colors():
    colors_arr = np.asarray(pcd.colors)
    is_black = np.all(colors_arr <= 0.02, axis=-1)
    if np.sum(is_black) > 0:
        print(f" -> Filtered {np.sum(is_black):,} black background border points.")
        pcd = pcd.select_by_index(np.where(~is_black)[0])

# 2. Statistical Outlier Removal
clean_pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=30, std_ratio=1.5)
print(f" -> Inlier points: {len(clean_pcd.points):,}")

# 3. Adaptive voxel downsampling if >2M points
if len(clean_pcd.points) > 2_000_000:
    print(" -> Downsampling point cloud to optimal 1.5M points for Poisson stability...")
    clean_pcd = clean_pcd.voxel_down_sample(voxel_size=0.003)
    print(f" -> Working point cloud: {len(clean_pcd.points):,} points")

# 4. Screened Poisson Surface Reconstruction (Depth 9)
mesh_hd, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(clean_pcd, depth=9)
print(f" -> Initial Mesh: {len(mesh_hd.vertices):,} vertices, {len(mesh_hd.triangles):,} faces")

# 5. Trim low-density boundary envelope
densities_arr = np.asarray(densities)
threshold = np.quantile(densities_arr, 0.03)
mesh_hd.remove_vertices_by_mask(densities_arr < threshold)

# 6. Clean surface geometry
mesh_hd.remove_duplicated_vertices()
mesh_hd.remove_duplicated_triangles()
mesh_hd.remove_degenerate_triangles()
mesh_hd.remove_non_manifold_edges()

# 7. Taubin non-shrinking smoothing to preserve bamboo ribs
mesh_hd = mesh_hd.filter_smooth_taubin(number_of_iterations=3)
mesh_hd.compute_vertex_normals()

hd_mesh_ply = dense_dir / "dense_mesh_hd.ply"
o3d.io.write_triangle_mesh(str(hd_mesh_ply), mesh_hd)
print(f"🎉 HD Dense Mesh saved: {hd_mesh_ply} ({len(mesh_hd.vertices):,} vertices, {len(mesh_hd.triangles):,} faces)")

# 8. Downsample dense point cloud for easy local viewing on Mac
pcd_view = clean_pcd.voxel_down_sample(voxel_size=0.005)
pcd_view_ply = dense_dir / "dense_preview.ply"
o3d.io.write_point_cloud(str(pcd_view_ply), pcd_view)

## 🎁 Step 6: Compress Results & Download
Creates `dense_mvs_results.zip` in `/kaggle/working/` and provides a direct download link.

In [ ]:
import os
from IPython.display import FileLink, display

out_zip = Path("/kaggle/working/dense_mvs_results.zip")
files_to_zip = [
    str(hd_mesh_ply),
    str(pcd_view_ply)
]
valid_files = " ".join([f for f in files_to_zip if f and os.path.exists(f)])
!zip -j "{out_zip}" {valid_files}

size_mb = os.path.getsize(out_zip) / (1024 * 1024)
print(f"\n🎉 ALL STEPS COMPLETE! Archive created: {out_zip.name} ({size_mb:.1f} MB)")
print("Click the link below to download your dense 3D model, or download it from the right sidebar under 'Output':\n")
display(FileLink("dense_mvs_results.zip"))